In [2]:
"""
MHIPEX — Three Missing Ablation Experiments for SCI Journal Submission
Run on Kaggle with GPU T4 x2 + Internet enabled
Estimated runtime: ~90 minutes total

Experiment 1: Focal Loss vs. Weighted CE
Experiment 2: Unweighted CE vs. Weighted CE
Experiment 3: Ensemble Weight Sensitivity (β sweep, post-hoc, no retraining)

Usage: Copy into a Kaggle notebook. Make sure Cell 1 (data download + preprocessing)
       from kaggle_mhipex_kg.py has already been run, so that proc/train_v12.jsonl
       and proc/dev_v12.jsonl exist.
"""

import json, os, gc, warnings, re
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from sklearn.metrics import recall_score, classification_report
from pathlib import Path
from itertools import product as iterproduct

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ── Paths ──
PROC_DIR = Path("proc")
OUT_DIR = Path("out_ablations")
OUT_DIR.mkdir(exist_ok=True)

AT_MAP = {"FALSE": 0, "PROBABLE": 1, "TRUE": 2}
ISAT_MAP = {"FALSE": 0, "TRUE": 1}
AT_NAMES = ["FALSE", "PROBABLE", "TRUE"]
ISAT_NAMES = ["FALSE", "TRUE"]
SPECIAL_TOKENS = ["<P>", "</P>", "<L>", "</L>", "<DATE>", "</DATE>", "<LANG>", "</LANG>"]

MODEL_NAME = "dbmdz/bert-base-historic-multilingual-cased"

# ══════════════════════════════════════════════════════════════════
#  Data Download & V12 Preprocessing
# ══════════════════════════════════════════════════════════════════
import urllib.request

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
PROC_DIR.mkdir(exist_ok=True)

BASE_URL = "https://raw.githubusercontent.com/hipe-eval/HIPE-2026-data/main/data/sandbox"
FILES = {
    "en-train": f"{BASE_URL}/en-train.jsonl", "fr-train": f"{BASE_URL}/fr-train.jsonl", "de-train": f"{BASE_URL}/de-train.jsonl",
    "en-dev":   f"{BASE_URL}/en-dev.jsonl",   "fr-dev":   f"{BASE_URL}/fr-dev.jsonl",   "de-dev":   f"{BASE_URL}/de-dev.jsonl",
}

print("\n── Downloading Data ──")
for name, url in FILES.items():
    dst = DATA_DIR / f"{name}.jsonl"
    if not dst.exists():
        print(f"  Downloading {name}.jsonl ...")
        urllib.request.urlretrieve(url, dst)

def clean_text(t, max_chars=850):
    return re.sub(r"\s+", " ", re.sub(r"[\x00-\x1f\x7f-\x9f]", " ", t)).strip()[:max_chars]

def build_input_v12(text, pers_list, loc_list, date_str="", lang=""):
    p = " ; ".join(clean_text(m, 100) for m in pers_list) if pers_list else "UNKNOWN"
    l = " ; ".join(clean_text(m, 100) for m in loc_list)  if loc_list  else "UNKNOWN"
    date_tok = f"<DATE> {date_str} </DATE> " if date_str else ""
    lang_tok = f"<LANG> {lang} </LANG> "     if lang     else ""
    return f"<P> {p} </P> <L> {l} </L> {date_tok}{lang_tok}{clean_text(text)}"

def load_and_process(path, lang):
    records = []
    for line in open(path, encoding="utf-8"):
        doc = json.loads(line)
        date_str = str(doc.get("date", ""))[:10]
        for pair in doc.get("sampled_pairs", []):
            at_raw = pair.get("at", "FALSE")
            isat_raw = pair.get("isAt", "FALSE")
            if at_raw not in AT_MAP or isat_raw not in ISAT_MAP: continue
            records.append({
                "text": build_input_v12(doc["text"], pair["pers_mentions_list"], pair["loc_mentions_list"], date_str, lang),
                "at_label": at_raw, "isat_label": isat_raw,
                "pers_qid": pair.get("pers_wikidata_qid", pair.get("pers_qid", "")),
                "loc_qid": pair.get("loc_wikidata_qid", pair.get("loc_qid", "")),
                "lang": lang, "doc_id": doc["document_id"],
            })
    return records

print("\n── Preprocessing V12 Format ──")
for split in ["train", "dev"]:
    out_path = PROC_DIR / f"{split}_v12.jsonl"
    if not out_path.exists():
        all_recs = []
        for lang in ["en", "fr", "de"]:
            all_recs.extend(load_and_process(DATA_DIR / f"{lang}-{split}.jsonl", lang))
        with open(out_path, "w", encoding="utf-8") as f:
            for r in all_recs: f.write(json.dumps(r, ensure_ascii=False) + "\n")
        print(f"  Processed {split}: {len(all_recs)} pairs")

# ══════════════════════════════════════════════════════════════════
#  Shared Components
# ══════════════════════════════════════════════════════════════════

class HIPEDataset(Dataset):
    def __init__(self, path, tokenizer, max_len=256):
        self.data = [json.loads(l) for l in open(path, encoding="utf-8")]
        self.tok = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        d = self.data[idx]
        enc = self.tok(d["text"], max_length=self.max_len,
                       truncation=True, padding="max_length", return_tensors="pt")
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "at_label": AT_MAP[d["at_label"]],
            "isat_label": ISAT_MAP[d["isat_label"]],
        }


class MHIPEXClassifier(nn.Module):
    def __init__(self, model_name, n_at=3, n_isat=2, dropout=0.15, n_drops=3):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        h = self.encoder.config.hidden_size
        self.n_drops = n_drops
        self.dropouts = nn.ModuleList([nn.Dropout(dropout) for _ in range(n_drops)])
        self.at_head = nn.Linear(h, n_at)
        self.isat_head = nn.Linear(h, n_isat)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_out = out.last_hidden_state[:, 0]
        mask = attention_mask.unsqueeze(-1).float()
        mean_out = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1)
        h = 0.5 * cls_out + 0.5 * mean_out

        at_logits_sum = torch.zeros(h.size(0), 3, device=h.device)
        isat_logits_sum = torch.zeros(h.size(0), 2, device=h.device)
        for drop in self.dropouts:
            hd = drop(h)
            at_logits_sum += self.at_head(hd)
            isat_logits_sum += self.isat_head(hd)
        return at_logits_sum / self.n_drops, isat_logits_sum / self.n_drops


class FocalLoss(nn.Module):
    """Focal Loss (Lin et al., 2017) with class weights."""
    def __init__(self, weight=None, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        self.weight = weight

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction="none")
        pt = torch.exp(-ce)
        focal = ((1 - pt) ** self.gamma) * ce
        return focal.mean()


def train_and_eval(tag, at_loss_fn, isat_loss_fn, epochs=25, lr=8e-6, bs=16):
    """Train hmBERT with given loss functions, return best dev MR + saved probs."""
    print(f"\n{'='*60}")
    print(f"  Training: {tag}")
    print(f"{'='*60}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenizer.add_special_tokens({"additional_special_tokens": SPECIAL_TOKENS})

    model = MHIPEXClassifier(MODEL_NAME).to(DEVICE)
    model.encoder.resize_token_embeddings(len(tokenizer))

    train_ds = HIPEDataset(PROC_DIR / "train_v12.jsonl", tokenizer)
    dev_ds = HIPEDataset(PROC_DIR / "dev_v12.jsonl", tokenizer)
    train_dl = DataLoader(train_ds, batch_size=bs, shuffle=True, num_workers=2)
    dev_dl = DataLoader(dev_ds, batch_size=bs * 2, num_workers=2)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(train_dl) * epochs
    scheduler = get_cosine_schedule_with_warmup(optimizer, int(0.12 * total_steps), total_steps)

    best_mr = 0
    patience, max_patience = 0, 8

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in train_dl:
            ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            at_y = batch["at_label"].to(DEVICE)
            isat_y = batch["isat_label"].to(DEVICE)

            at_logits, isat_logits = model(ids, mask)
            loss = at_loss_fn(at_logits, at_y) + isat_loss_fn(isat_logits, isat_y)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        # Evaluate
        model.eval()
        all_at_true, all_at_pred = [], []
        all_is_true, all_is_pred = [], []
        all_at_probs, all_is_probs = [], []

        with torch.no_grad():
            for batch in dev_dl:
                ids = batch["input_ids"].to(DEVICE)
                mask = batch["attention_mask"].to(DEVICE)
                at_logits, isat_logits = model(ids, mask)
                at_probs = torch.softmax(at_logits, dim=-1)
                is_probs = torch.softmax(isat_logits, dim=-1)

                all_at_true.extend(batch["at_label"].tolist())
                all_at_pred.extend(at_probs.argmax(dim=-1).cpu().tolist())
                all_is_true.extend(batch["isat_label"].tolist())
                all_is_pred.extend(is_probs.argmax(dim=-1).cpu().tolist())
                all_at_probs.extend(at_probs.cpu().numpy())
                all_is_probs.extend(is_probs.cpu().numpy())

        at_mr = recall_score(all_at_true, all_at_pred, average="macro", zero_division=0)
        is_mr = recall_score(all_is_true, all_is_pred, average="macro", zero_division=0)
        mr = round((at_mr + is_mr) / 2, 4)

        print(f"  Epoch {epoch+1:2d} | Loss: {total_loss/len(train_dl):.4f} | MR: {mr:.4f} (at={at_mr:.4f}, isAt={is_mr:.4f})")

        if mr > best_mr:
            best_mr = mr
            patience = 0
            save_dir = OUT_DIR / tag
            save_dir.mkdir(exist_ok=True)
            torch.save({
                "probs_at": torch.tensor(np.array(all_at_probs)),
                "probs_isat": torch.tensor(np.array(all_is_probs)),
                "at_true": all_at_true, "is_true": all_is_true,
                "at_pred": all_at_pred, "is_pred": all_is_pred,
            }, save_dir / "best_probs.pt")
        else:
            patience += 1
            if patience >= max_patience:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    print(f"  Best MR: {best_mr:.4f}")
    del model, optimizer
    gc.collect()
    torch.cuda.empty_cache()
    return best_mr


def calibrate(tag):
    """Post-hoc threshold calibration on saved probabilities."""
    data = torch.load(OUT_DIR / tag / "best_probs.pt", weights_only=True)
    probs_at = data["probs_at"].numpy()
    probs_isat = data["probs_isat"].numpy()
    at_true = data["at_true"]
    is_true = data["is_true"]

    # AT calibration
    best_at_mr, best_at_preds = 0, []
    prob_range = np.arange(0.20, 0.55, 0.05)
    for t_prob, t_true in iterproduct(prob_range, prob_range):
        preds = []
        for p in probs_at:
            if p[2] >= t_true: preds.append(2)
            elif p[1] >= t_prob: preds.append(1)
            else: preds.append(0)
        mr = recall_score(at_true, preds, average="macro", zero_division=0)
        if mr > best_at_mr:
            best_at_mr = mr
            best_at_preds = preds

    # isAt calibration
    best_isat_mr = 0
    for t in np.arange(0.15, 0.60, 0.05):
        preds = []
        for i, p in enumerate(probs_isat):
            if best_at_preds[i] == 0: preds.append(0)
            elif p[1] >= t: preds.append(1)
            else: preds.append(0)
        mr = recall_score(is_true, preds, average="macro", zero_division=0)
        if mr > best_isat_mr:
            best_isat_mr = mr

    total_mr = round((best_at_mr + best_isat_mr) / 2, 4)
    return {"mr": total_mr, "at_mr": round(best_at_mr, 4), "isat_mr": round(best_isat_mr, 4)}


# ══════════════════════════════════════════════════════════════════
#  EXPERIMENT 1: Focal Loss vs. Weighted CE
# ══════════════════════════════════════════════════════════════════

print("\n" + "=" * 64)
print("  EXPERIMENT 1: Loss Function Comparison")
print("=" * 64)

# 1a: Weighted CE (our default)
at_w = torch.tensor([0.80, 1.50, 2.40], device=DEVICE)
isat_w = torch.tensor([0.70, 2.60], device=DEVICE)
at_ce = nn.CrossEntropyLoss(weight=at_w)
isat_ce = nn.CrossEntropyLoss(weight=isat_w)
train_and_eval("weighted_ce", at_ce, isat_ce)

# 1b: Focal Loss (gamma=2, same class weights)
at_focal = FocalLoss(weight=at_w.clone(), gamma=2.0)
isat_focal = FocalLoss(weight=isat_w.clone(), gamma=2.0)
train_and_eval("focal_loss", at_focal, isat_focal)


# ══════════════════════════════════════════════════════════════════
#  EXPERIMENT 2: Unweighted CE vs. Weighted CE
# ══════════════════════════════════════════════════════════════════

print("\n" + "=" * 64)
print("  EXPERIMENT 2: Class Weight Ablation")
print("=" * 64)

# 2a: Unweighted CE
at_unw = nn.CrossEntropyLoss()
isat_unw = nn.CrossEntropyLoss()
train_and_eval("unweighted_ce", at_unw, isat_unw)

# (Weighted CE was already trained as "weighted_ce" above)


# ══════════════════════════════════════════════════════════════════
#  EXPERIMENT 3: Ensemble Weight Sensitivity (Post-hoc)
# ══════════════════════════════════════════════════════════════════

print("\n" + "=" * 64)
print("  EXPERIMENT 3: Ensemble β Sensitivity")
print("=" * 64)

# Load saved probabilities from the MAIN experiments (out/ directory from original training)
# If those don't exist, use the weighted_ce run from Exp 1 + create an XLM-R placeholder
MAIN_OUT = Path("out")
hmbert_path = MAIN_OUT / "hmbert_v12" / "best_probs.pt"
xlmr_path = MAIN_OUT / "xlmr_v12" / "best_probs.pt"

if hmbert_path.exists() and xlmr_path.exists():
    hm_data = torch.load(hmbert_path, weights_only=True)
    xr_data = torch.load(xlmr_path, weights_only=True)

    hm_at = hm_data["probs_at"].numpy()
    hm_is = hm_data["probs_isat"].numpy()
    xr_at = xr_data["probs_at"].numpy()
    xr_is = xr_data["probs_isat"].numpy()
    at_true = hm_data["at_true"]
    is_true = hm_data["is_true"]

    betas = [0.30, 0.40, 0.50, 0.60, 0.70, 0.80]
    beta_results = []

    for beta in betas:
        ens_at = beta * hm_at + (1 - beta) * xr_at
        ens_is = beta * hm_is + (1 - beta) * xr_is

        # Calibrate
        best_at_mr, best_at_preds = 0, []
        prob_range = np.arange(0.20, 0.55, 0.05)
        for t_prob, t_true in iterproduct(prob_range, prob_range):
            preds = []
            for p in ens_at:
                if p[2] >= t_true: preds.append(2)
                elif p[1] >= t_prob: preds.append(1)
                else: preds.append(0)
            mr = recall_score(at_true, preds, average="macro", zero_division=0)
            if mr > best_at_mr:
                best_at_mr = mr
                best_at_preds = preds

        best_isat_mr = 0
        for t in np.arange(0.15, 0.60, 0.05):
            preds = []
            for i, p in enumerate(ens_is):
                if best_at_preds[i] == 0: preds.append(0)
                elif p[1] >= t: preds.append(1)
                else: preds.append(0)
            mr = recall_score(is_true, preds, average="macro", zero_division=0)
            if mr > best_isat_mr:
                best_isat_mr = mr

        total = round((best_at_mr + best_isat_mr) / 2, 4)
        beta_results.append((beta, total, round(best_at_mr, 4), round(best_isat_mr, 4)))
        print(f"  β={beta:.2f} | MR: {total:.4f} | at: {best_at_mr:.4f} | isAt: {best_isat_mr:.4f}")

    print("\n  β Sensitivity Table:")
    print(f"  {'β':>5s}  {'MR':>7s}  {'at':>7s}  {'isAt':>7s}")
    for b, mr, at, isat in beta_results:
        marker = " *" if b == 0.60 else ""
        print(f"  {b:5.2f}  {mr:7.4f}  {at:7.4f}  {isat:7.4f}{marker}")
else:
    print("  WARNING: Main experiment probs not found in out/")
    print("  Run the original training pipeline first (kaggle_mhipex_v12_cell1.py + cell2.py)")
    print("  Then re-run this script for Experiment 3.")


# ══════════════════════════════════════════════════════════════════
#  Results Summary
# ══════════════════════════════════════════════════════════════════

print("\n" + "=" * 64)
print("  ABLATION RESULTS SUMMARY")
print("=" * 64)

# Calibrate all trained models
for tag, label in [
    ("weighted_ce", "Weighted CE"),
    ("focal_loss", "Focal Loss (γ=2)"),
    ("unweighted_ce", "Unweighted CE"),
]:
    path = OUT_DIR / tag / "best_probs.pt"
    if path.exists():
        cal = calibrate(tag)
        print(f"  {label:25s} | MR: {cal['mr']:.4f} | at: {cal['at_mr']:.4f} | isAt: {cal['isat_mr']:.4f}")

# Save results
import csv
csv_path = OUT_DIR / "ablation_results.csv"
with open(csv_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["Experiment", "MR", "at_recall", "isAt_recall"])
    for tag, label in [("weighted_ce", "Weighted CE"), ("focal_loss", "Focal Loss"), ("unweighted_ce", "Unweighted CE")]:
        path = OUT_DIR / tag / "best_probs.pt"
        if path.exists():
            cal = calibrate(tag)
            w.writerow([label, cal["mr"], cal["at_mr"], cal["isat_mr"]])

print(f"\n  Results saved to: {csv_path}")
print(f"  Download the 'out_ablations/' folder from Kaggle output!")


Device: cuda

── Downloading Data ──

── Preprocessing V12 Format ──
  Processed train: 6170 pairs
  Processed dev: 2081 pairs

  EXPERIMENT 1: Loss Function Comparison

  Training: weighted_ce


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: dbmdz/bert-base-historic-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Epoch  1 | Loss: 1.7072 | MR: 0.4184 (at=0.3369, isAt=0.5000)
  Epoch  2 | Loss: 1.6590 | MR: 0.4698 (at=0.4254, isAt=0.5141)
  Epoch  3 | Loss: 1.5645 | MR: 0.4560 (at=0.4018, isAt=0.5102)
  Epoch  4 | Loss: 1.4281 | MR: 0.5324 (at=0.4603, isAt=0.6044)
  Epoch  5 | Loss: 1.3204 | MR: 0.5539 (at=0.4705, isAt=0.6374)
  Epoch  6 | Loss: 1.1565 | MR: 0.5462 (at=0.4414, isAt=0.6510)
  Epoch  7 | Loss: 1.0177 | MR: 0.5076 (at=0.4427, isAt=0.5725)
  Epoch  8 | Loss: 0.9402 | MR: 0.5586 (at=0.4660, isAt=0.6511)
  Epoch  9 | Loss: 0.8071 | MR: 0.5241 (at=0.4646, isAt=0.5836)
  Epoch 10 | Loss: 0.7116 | MR: 0.5425 (at=0.4622, isAt=0.6228)
  Epoch 11 | Loss: 0.6249 | MR: 0.5138 (at=0.4492, isAt=0.5784)
  Epoch 12 | Loss: 0.5954 | MR: 0.5140 (at=0.4555, isAt=0.5724)
  Epoch 13 | Loss: 0.4976 | MR: 0.5056 (at=0.4404, isAt=0.5709)
  Epoch 14 | Loss: 0.4652 | MR: 0.5110 (at=0.4441, isAt=0.5779)
  Epoch 15 | Loss: 0.3891 | MR: 0.5074 (at=0.4358, isAt=0.5789)
  Epoch 16 | Loss: 0.3685 | MR: 0.5186 (

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: dbmdz/bert-base-historic-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Epoch  1 | Loss: 0.9971 | MR: 0.4152 (at=0.3348, isAt=0.4956)
  Epoch  2 | Loss: 0.9644 | MR: 0.5047 (at=0.4352, isAt=0.5741)
  Epoch  3 | Loss: 0.9018 | MR: 0.5360 (at=0.4361, isAt=0.6359)
  Epoch  4 | Loss: 0.8059 | MR: 0.5430 (at=0.4479, isAt=0.6380)
  Epoch  5 | Loss: 0.6932 | MR: 0.5449 (at=0.4701, isAt=0.6198)
  Epoch  6 | Loss: 0.5960 | MR: 0.5679 (at=0.4660, isAt=0.6697)
  Epoch  7 | Loss: 0.5135 | MR: 0.5640 (at=0.4815, isAt=0.6465)
  Epoch  8 | Loss: 0.4461 | MR: 0.5001 (at=0.4217, isAt=0.5784)
  Epoch  9 | Loss: 0.4004 | MR: 0.5507 (at=0.4742, isAt=0.6272)
  Epoch 10 | Loss: 0.3359 | MR: 0.5419 (at=0.4608, isAt=0.6230)
  Epoch 11 | Loss: 0.3035 | MR: 0.5524 (at=0.4669, isAt=0.6379)
  Epoch 12 | Loss: 0.2675 | MR: 0.5323 (at=0.4642, isAt=0.6004)
  Epoch 13 | Loss: 0.2371 | MR: 0.5404 (at=0.4437, isAt=0.6371)
  Epoch 14 | Loss: 0.1977 | MR: 0.5332 (at=0.4553, isAt=0.6111)
  Early stopping at epoch 14
  Best MR: 0.5679

  EXPERIMENT 2: Class Weight Ablation

  Training: unwei

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: dbmdz/bert-base-historic-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Epoch  1 | Loss: 1.3925 | MR: 0.4167 (at=0.3333, isAt=0.5000)
  Epoch  2 | Loss: 1.2645 | MR: 0.4213 (at=0.3426, isAt=0.5000)
  Epoch  3 | Loss: 1.2065 | MR: 0.4219 (at=0.3438, isAt=0.5000)
  Epoch  4 | Loss: 1.1340 | MR: 0.5050 (at=0.4437, isAt=0.5663)
  Epoch  5 | Loss: 1.0350 | MR: 0.4971 (at=0.4389, isAt=0.5553)
  Epoch  6 | Loss: 0.9213 | MR: 0.4762 (at=0.4091, isAt=0.5433)
  Epoch  7 | Loss: 0.8361 | MR: 0.5231 (at=0.4591, isAt=0.5871)
  Epoch  8 | Loss: 0.7365 | MR: 0.5459 (at=0.4674, isAt=0.6244)
  Epoch  9 | Loss: 0.6540 | MR: 0.5475 (at=0.4694, isAt=0.6256)
  Epoch 10 | Loss: 0.5765 | MR: 0.5153 (at=0.4438, isAt=0.5868)
  Epoch 11 | Loss: 0.5086 | MR: 0.5101 (at=0.4420, isAt=0.5782)
  Epoch 12 | Loss: 0.4495 | MR: 0.5403 (at=0.4521, isAt=0.6285)
  Epoch 13 | Loss: 0.4038 | MR: 0.5189 (at=0.4436, isAt=0.5941)
  Epoch 14 | Loss: 0.3613 | MR: 0.5057 (at=0.4337, isAt=0.5777)
  Epoch 15 | Loss: 0.3272 | MR: 0.5257 (at=0.4481, isAt=0.6033)
  Epoch 16 | Loss: 0.2868 | MR: 0.5102 (